In [1]:
import pickle
import random
import os

In [2]:
base_dir = "../Lab4"
models = {
    "unigram": pickle.load(open(os.path.join(base_dir, "unigram_model.pkl"), "rb")),
    "bigram": pickle.load(open(os.path.join(base_dir, "bigram_model.pkl"), "rb")),
    "trigram": pickle.load(open(os.path.join(base_dir, "trigram_model.pkl"), "rb")),
    "quadgram": pickle.load(open(os.path.join(base_dir, "quad_model.pkl"), "rb")),
}

In [3]:
def tok2str(tok):
    """Convert tuple tokens or plain strings to readable strings"""
    if isinstance(tok, tuple):
        return " ".join(str(t) for t in tok)
    return str(tok)

In [4]:
def get_random_context(model):
    """Pick a random valid context from the model"""
    return random.choice(list(model.keys())) if model else ()

In [5]:
def greedy_generate(model, n, max_len=20, end_token="</s>"):
    if n == 1:  # unigram
        sentence = []
        for _ in range(max_len):
            if not model: break
            next_word = max(model, key=model.get)
            if tok2str(next_word) == end_token:
                break
            sentence.append(tok2str(next_word))
        return " ".join(sentence) if sentence else "<EMPTY>"

    # start with a random context
    context = get_random_context(model)
    sentence = list(context)

    for _ in range(max_len):
        candidates = model.get(context, {})
        candidates = {w: p for w, p in candidates.items() if isinstance(p, (float, int))}
        if not candidates:
            break
        next_word = max(candidates, key=candidates.get)
        if tok2str(next_word) == end_token:
            break
        sentence.append(next_word)
        context = tuple(sentence[-(n - 1):])

    output = [tok2str(tok) for tok in sentence[n - 1:]]
    return " ".join(output) if output else "<EMPTY>"



In [6]:
def beam_search_generate(model, n, beam_size=20, max_len=20, end_token="</s>"):
    if n == 1:  # unigram
        beams = [([], 1.0)]
        for _ in range(max_len):
            new_beams = []
            for seq, score in beams:
                for word, prob in model.items():
                    if not isinstance(prob, (float, int)):
                        continue
                    new_beams.append((seq + [tok2str(word)], score * prob))
            if not new_beams:
                return " ".join(beams[0][0]) if beams else "<EMPTY>"
            beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_size]
        return " ".join(beams[0][0]) if beams else "<EMPTY>"

    # start with random context
    context = get_random_context(model)
    beams = [(list(context), 1.0)]

    for _ in range(max_len):
        new_beams = []
        for seq, score in beams:
            context = tuple(seq[-(n - 1):])
            candidates = model.get(context, {})
            for word, prob in candidates.items():
                if not isinstance(prob, (float, int)):
                    continue
                new_seq = seq + [word]
                new_score = score * prob
                new_beams.append((new_seq, new_score))
        if not new_beams:
            break
        beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_size]
        if any(tok2str(seq[-1]) == end_token for seq, _ in beams):
            break

    if not beams:
        return "<EMPTY>"

    best_seq, _ = beams[0]
    output = [tok2str(tok) for tok in best_seq[n - 1:]]
    return " ".join(output) if output else "<EMPTY>"

In [7]:
for name, model in models.items():
    n = {"unigram": 1, "bigram": 2, "trigram": 3, "quadgram": 4}[name]

    print(f"\n=== {name.upper()} MODEL ===")

    print("\nGreedy Samples:")
    for i in range(5):  # set to 100 for full run
        print(f"{i+1}: {greedy_generate(model, n)}")

    print("\nBeam Search Samples:")
    for i in range(5):  # set to 100 for full run
        print(f"{i+1}: {beam_search_generate(model, n)}")


=== UNIGRAM MODEL ===

Greedy Samples:
1:                    
2:                    
3:                    
4:                    
5:                    

Beam Search Samples:
1: 
2: 
3: 
4: 
5: 

=== BIGRAM MODEL ===

Greedy Samples:
1: के लिए एक बार फिर से पहले ही नहीं है। इस दौरान उन्होंने कहा कि वह अपने घर में भी
2: से पहले ही नहीं है। इस दौरान उन्होंने कहा कि वह अपने घर में भी नहीं है। इस दौरान उन्होंने
3: नहीं है। इस दौरान उन्होंने कहा कि वह अपने घर में भी नहीं है। इस दौरान उन्होंने कहा कि वह
4: के लिए एक बार फिर से पहले ही नहीं है। इस दौरान उन्होंने कहा कि वह अपने घर में भी
5: की ओर से पहले ही नहीं है। इस दौरान उन्होंने कहा कि वह अपने घर में भी नहीं है। इस

Beam Search Samples:
1: का कहना है . . . . . . . . . . . . . . उन्होंने कहा कि
2: ‍ होंने कहा गया है . . . . . . . . . . . . . . .
3: ओपन सुपर किंग्स इलेवन पंजाब किंग्स इलेवन पंजाब सरकार ने बताया जा रहा है . . . . . .
4: पानी की मौत हो सकता है . . . . . . . . . . . . . .
5: का आरोप लगाया जा रहा है . . . . . . . . . . . . . .